In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("lakehouse.duckdb")
con.execute("SHOW TABLES").df()

,name
0,raw_customers
1,raw_orders
2,raw_products
3,stage_customers
4,stage_orders
5,stage_products


In [2]:
# DIM_CLIENTE: atributos descriptivos del cliente
con.execute("""
    CREATE OR REPLACE TABLE dim_cliente AS
    SELECT customer_id, first_name, last_name, email, phone, city, 
           country, age, registration_date, loyalty_tier
    FROM stage_customers
""")

# DIM_PRODUCTO: atributos descriptivos del producto
con.execute("""
    CREATE OR REPLACE TABLE dim_producto AS
    SELECT product_id, product_name, category, price_usd, cost_usd, 
           stock_units, supplier, active
    FROM stage_products
""")

# FACT_VENTAS: tablón a nivel de transacción (cada fila = un pedido),
# con las llaves hacia las dimensiones y las métricas de la venta.
# No se necesita JOIN porque stage_orders ya tiene los IDs validados
# contra dim_cliente y dim_producto desde el Paso 2.
con.execute("""
    CREATE OR REPLACE TABLE fact_ventas AS
    SELECT 
        order_id,
        customer_id,
        product_id,
        quantity,
        unit_price_usd,
        total_amount_usd,
        discount_pct,
        order_date,
        ship_date,
        status,
        payment_method
    FROM stage_orders
""")

con.execute("SHOW TABLES").df()

,name
0,dim_cliente
1,dim_producto
2,fact_ventas
3,raw_customers
4,raw_orders
5,raw_products
6,stage_customers
7,stage_orders
8,stage_products


In [3]:
con.close()

## Esquema de la capa ANALYTICS

### DIM_CLIENTE
| Columna | Tipo | Descripción |
|---|---|---|
| customer_id | INTEGER | Identificador único del cliente |
| first_name | VARCHAR | Nombre del cliente |
| last_name | VARCHAR | Apellido del cliente |
| email | VARCHAR | Correo electrónico |
| phone | VARCHAR | Teléfono de contacto |
| city | VARCHAR | Ciudad de residencia |
| country | VARCHAR | País (único valor: Colombia) |
| age | INTEGER | Edad del cliente |
| registration_date | DATE | Fecha de registro |
| loyalty_tier | VARCHAR | Nivel de fidelidad (GOLD, SILVER, BRONZE, NO DETERMINADO) |

### DIM_PRODUCTO
| Columna | Tipo | Descripción |
|---|---|---|
| product_id | INTEGER | Identificador único del producto |
| product_name | VARCHAR | Nombre del producto |
| category | VARCHAR | Categoría del producto |
| price_usd | DOUBLE | Precio de venta en USD |
| cost_usd | DOUBLE | Costo del producto en USD |
| stock_units | DOUBLE | Unidades disponibles en stock |
| supplier | VARCHAR | Proveedor (formato "PROVEEDOR [LETRA]") |
| active | BOOLEAN | Indica si el producto está activo |

### FACT_VENTAS
| Columna | Tipo | Descripción |
|---|---|---|
| order_id | INTEGER | Identificador único del pedido |
| customer_id | INTEGER | FK a DIM_CLIENTE |
| product_id | INTEGER | FK a DIM_PRODUCTO |
| quantity | INTEGER | Cantidad de unidades pedidas |
| unit_price_usd | DOUBLE | Precio unitario al momento de la venta |
| total_amount_usd | DOUBLE | Monto total del pedido |
| discount_pct | DOUBLE | Porcentaje de descuento aplicado |
| order_date | DATE | Fecha del pedido |
| ship_date | DATE | Fecha de envío |
| status | VARCHAR | Estado del pedido (COMPLETED, PENDING, CANCELLED, NO DETERMINADO) |
| payment_method | VARCHAR | Método de pago utilizado |